# Responsibilities Of An AI Engineer

Run `4-classify-job-postings.ipynb` first so that `jobs/2-classified-jobs.jsonl` exists.

This notebook reads the LLM-filtered AI engineer job postings and uses `gpt-5.4-mini` to summarize the responsibilities that consistently appear across the roles.


In [1]:
import re
from pathlib import Path

import pandas as pd
from openai import OpenAI
from IPython.display import Markdown, display

classified_jobs_path = Path("jobs") / "2-classified-jobs.jsonl"

if not classified_jobs_path.exists():
    raise FileNotFoundError(
        f"Classified jobs JSONL file not found: {classified_jobs_path.resolve()}. Run 4-classify-job-postings.ipynb first."
    )

jobs_for_summary = pd.read_json(classified_jobs_path, lines=True)

if jobs_for_summary.empty:
    raise ValueError(
        "The LLM-filtered jobs JSONL file is empty. Run 4-classify-job-postings.ipynb first."
    )

if "description" not in jobs_for_summary.columns:
    raise KeyError(
        "The LLM-filtered jobs JSONL file must contain a 'description' column."
    )

jobs_for_summary = jobs_for_summary[jobs_for_summary["description"].notna()].copy()
jobs_for_summary["description"] = (
    jobs_for_summary["description"].astype(str).str.strip()
)
jobs_for_summary = jobs_for_summary[jobs_for_summary["description"] != ""].copy()

if "job_url" in jobs_for_summary.columns:
    jobs_for_summary = jobs_for_summary.drop_duplicates(subset=["job_url"])
else:
    jobs_for_summary = jobs_for_summary.drop_duplicates(
        subset=["title", "company", "description"]
    )

if jobs_for_summary.empty:
    raise ValueError(
        "No usable job descriptions remain after cleaning and deduplication."
    )


def clean_description(text):
    text = str(text)
    text = text.replace("**", "")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


max_description_chars = 10000
job_entries = []

for _, job in jobs_for_summary.iterrows():
    title = "" if pd.isna(job.get("title")) else str(job.get("title")).strip()
    company = "" if pd.isna(job.get("company")) else str(job.get("company")).strip()
    description = clean_description(job.get("description", ""))

    if not description:
        continue

    if len(description) > max_description_chars:
        description = (
            description[:max_description_chars]
            + " [Description truncated for summarization.]"
        )

    job_entries.append(
        "\n".join(
            [
                f"Title: {title}",
                f"Company: {company}",
                f"Description: {description}",
            ]
        )
    )

if not job_entries:
    raise ValueError("No non-empty job descriptions were available for summarization.")

job_descriptions = "\n\n---\n\n".join(job_entries)

client = OpenAI()

instructions = """
You analyze AI engineering job postings and identify the shared responsibilities across them.

Focus only on responsibilities that appear repeatedly across the postings.
Ignore company marketing, benefits, equal opportunity text, hiring process details, and generic recruiting language.
Do not focus on responsibilities that only appear in one or two postings unless they are clearly central to nearly all of the roles.
""".strip()

prompt = f"""
Read the job descriptions below and summarize the common responsibilities these AI engineer roles are looking for.

Return markdown in exactly this format:
## Shared responsibilities
- 5 to 10 concise bullet points

Requirements:
- Focus on common responsibilities only
- Keep bullets concrete and specific
- Do not include commentary before or after the bullets
- Do not mention company names

Job descriptions:
{job_descriptions}
""".strip()

print(
    f"Summarizing responsibilities across {len(job_entries)} LLM-filtered job postings"
)
print(f"Combined description length: {len(job_descriptions):,} characters")

response = client.responses.create(
    model="gpt-5.4-mini",
    instructions=instructions,
    input=prompt,
    max_output_tokens=500,
    text={"verbosity": "low"},
)

summary = response.output_text
display(Markdown(summary))

Summarizing responsibilities across 18 LLM-filtered job postings
Combined description length: 98,243 characters


## Shared responsibilities
- Design, build, and deploy production AI/LLM applications and workflows
- Develop agentic systems with multi-step reasoning, tool use, and orchestration
- Integrate LLMs with enterprise APIs, data sources, and internal platforms
- Build retrieval, prompt engineering, and RAG-based solutions
- Create evaluation, testing, observability, and monitoring frameworks for AI systems
- Optimize AI applications for reliability, performance, latency, cost, and quality
- Collaborate with cross-functional stakeholders to translate business needs into technical solutions
- Produce reusable infrastructure, components, patterns, and documentation for team adoption
- Support production deployment, debugging, and iterative improvement of AI systems
- Mentor, guide, or influence other engineers and help establish best practices